# Notebook 01 — Sentinel-2 and NDVI: Anantapur

**Project:** Sentinel-2 vegetation analysis  
**Study area:** Anantapur, Andhra Pradesh, India  
**Acquisition date:** 9 October 2026 (from the QGIS layer name/metadata supplied)

## What this notebook teaches
1. What Sentinel-2 bands B04 and B08 measure.
2. Why red and near-infrared reflectance help identify vegetation.
3. How to inspect an exported NDVI GeoTIFF.
4. How to calculate valid-pixel statistics and plot a histogram.
5. How to interpret NDVI without overclaiming.

> **Important:** Your QGIS layer reports provider `virtualraster`. Export it as a real GeoTIFF first: right-click the NDVI layer → **Export → Save As…** → choose **GeoTIFF** and save it in your project folder. This makes it easier to analyze reliably in Python.


## 1. Sentinel-2 wavelengths

Sentinel-2 is a multispectral satellite: it measures reflected energy in several wavelength ranges.

| Band | Approximate wavelength range | Central wavelength (approx.) | Native pixel size | Typical use |
|---|---:|---:|---:|---|
| **B04 — Red** | 649–680 nm | 665 nm | 10 m | Chlorophyll absorbs much of the red light |
| **B08 — Near-infrared (NIR)** | 784–900 nm | 842 nm | 10 m | Healthy leaf structure often reflects strongly in NIR |
| B03 — Green | 543–578 nm | 560 nm | 10 m | True-colour composite |
| B02 — Blue | 458–523 nm | 490 nm | 10 m | True-colour composite |
| B05 — Red edge 1 | 698–713 nm | 705 nm | 20 m | Vegetation/chlorophyll-sensitive analyses |
| B06 — Red edge 2 | 733–748 nm | 740 nm | 20 m | Vegetation and red-edge analysis |
| B07 — Red edge 3 | 773–793 nm | 783 nm | 20 m | Vegetation and red-edge analysis |
| B8A — Narrow NIR | 855–875 nm | 865 nm | 20 m | Narrow NIR analyses |
| B11 — SWIR 1 | 1,565–1,655 nm | 1,610 nm | 20 m | Moisture and surface-material analysis |
| B12 — SWIR 2 | 2,100–2,280 nm | 2,190 nm | 20 m | Moisture and surface-material analysis |

**Unit reminder:** 1 µm = 1,000 nm. Visible light is roughly 400–700 nm. NIR is beyond red visible light and cannot be seen by human eyes. Band wavelength ranges are approximate; each band integrates energy over a range, not one exact wavelength.

### Why B04 and B08?
Green vegetation usually absorbs red light for photosynthesis and reflects relatively strongly in NIR. NDVI compares these responses. It is a useful signal, but it is not a direct measurement of biomass, yield, or plant health by itself.


## 2. NDVI formula and interpretation

\[
NDVI = \frac{NIR - Red}{NIR + Red}
= \frac{B08 - B04}{B08 + B04}
\]

NDVI is normally between **−1 and +1** when the denominator is valid.

| Approximate NDVI range | Common interpretation | Caution |
|---|---|---|
| Below 0 | Often water, cloud/shadow, or other non-vegetated surfaces | Not every negative pixel is water |
| 0 to 0.2 | Often bare soil, built-up surfaces, sparse cover, or mixed pixels | These classes overlap |
| 0.2 to 0.5 | Sparse-to-moderate vegetation is common | Thresholds vary with season, soil, and crop |
| Above 0.5 | Often dense green vegetation | Does not prove a specific crop or health state |

These are **rule-of-thumb ranges**, not universal class boundaries.

### Why did we change the QGIS display range to 0–0.5?
That was a **visualization choice** to stretch colours across a useful vegetation range. It does **not** recalculate the source NDVI values. Depending on the renderer, values below 0 or above 0.5 may display using endpoint colours. Statistical analysis should use the underlying pixels, not the display stretch.


In [ ]:
# If needed, install packages in the same Python environment as Jupyter:
# %pip install rasterio numpy matplotlib


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# EDIT THIS PATH to the GeoTIFF exported from QGIS.
# Replace YOUR_USERNAME and adjust the folder if your actual path differs.
ndvi_path = Path(r"C:\Users\YOUR_USERNAME\Documents\RemoteSensing_Projects\Project_01_Sentinel2_Vegetation\data\Anantapur_NDVI_20261009.tif")

print("NDVI file path:", ndvi_path)
print("File exists:", ndvi_path.exists())
if not ndvi_path.exists():
    print("Update ndvi_path to the actual location of your exported GeoTIFF before continuing.")


## 3. Inspect raster metadata

The QGIS information you shared reports CRS `EPSG:32643` (WGS 84 / UTM zone 43N), one floating-point band, and reported limits −1 to +1. This cell checks the **exported file itself**.


In [ ]:
import rasterio

if ndvi_path.exists():
    with rasterio.open(ndvi_path) as src:
        print("Driver:", src.driver)
        print("Width × height:", src.width, "×", src.height)
        print("Band count:", src.count)
        print("CRS:", src.crs)
        print("Pixel size:", src.res)
        print("Data type:", src.dtypes)
        print("NoData value:", src.nodata)
        print("Bounds:", src.bounds)
else:
    print("Skipped: set ndvi_path to an existing exported GeoTIFF.")


## 4. Read NDVI and create a valid-pixel mask

This excludes NoData and non-finite values. It cannot retrospectively prove that clouds, shadows, or zero denominators were handled correctly in the original QGIS calculation. Your screenshots show visible clouds, so cloud/shadow masking is an important next step.


In [ ]:
if ndvi_path.exists():
    with rasterio.open(ndvi_path) as src:
        ndvi_masked = src.read(1, masked=True).astype("float32")
        ndvi_values = np.asarray(ndvi_masked.filled(np.nan), dtype="float32")
        valid = np.isfinite(ndvi_values)

        if src.nodata is not None:
            valid &= ndvi_values != src.nodata

        outside_range = valid & ((ndvi_values < -1.0001) | (ndvi_values > 1.0001))
        print("Total pixels:", ndvi_values.size)
        print("Valid finite pixels:", int(valid.sum()))
        print("Pixels outside [-1, 1] tolerance:", int(outside_range.sum()))

        if valid.any():
            v = ndvi_values[valid]
            for label, value in [
                ("Minimum", np.min(v)),
                ("2nd percentile", np.percentile(v, 2)),
                ("25th percentile", np.percentile(v, 25)),
                ("Median", np.median(v)),
                ("75th percentile", np.percentile(v, 75)),
                ("98th percentile", np.percentile(v, 98)),
                ("Maximum", np.max(v)),
            ]:
                print(f"{label}: {value:.4f}")
else:
    print("Skipped: set ndvi_path to an existing exported GeoTIFF.")


## 5. Plot the NDVI histogram

The histogram shows how many valid pixels fall into each NDVI interval. It is a distribution of pixel values, not a map and not directly the percentage of land covered by vegetation.


In [ ]:
if ndvi_path.exists() and valid.any():
    plt.figure(figsize=(9, 5))
    plt.hist(ndvi_values[valid], bins=100)
    plt.xlabel("NDVI")
    plt.ylabel("Pixel count")
    plt.title("NDVI distribution — Anantapur, 9 October 2026")
    plt.xlim(-1, 1)
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()
else:
    print("Run the raster-loading cell first, after setting the correct path.")


## 6. Display NDVI using the full theoretical range

This uses −1 to +1 so negative values remain visible. It is separate from the QGIS 0–0.5 display stretch. This quick-look plot has no map layout, scale bar, or geographic labels.


In [ ]:
if ndvi_path.exists():
    with rasterio.open(ndvi_path) as src:
        show_arr = src.read(1, masked=True).astype("float32")
    plt.figure(figsize=(10, 8))
    im = plt.imshow(show_arr, vmin=-1, vmax=1, cmap="RdYlGn")
    plt.colorbar(im, label="NDVI")
    plt.title("NDVI — Anantapur, 9 October 2026")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped: set ndvi_path to an existing exported GeoTIFF.")


## 7. Optional: count pixels in broad NDVI intervals

These are exploratory bins, **not a validated land-cover classification**. Clouds, shadows, water, soil brightness, built-up surfaces, crop stage, irrigation, and mixed pixels can all affect NDVI.


In [ ]:
if ndvi_path.exists() and valid.any():
    bins = [-1.0, 0.0, 0.2, 0.5, 1.000001]
    labels = ["< 0", "0 to < 0.2", "0.2 to < 0.5", "0.5 to 1"]
    counts = np.histogram(ndvi_values[valid], bins=bins)[0]
    total = counts.sum()
    print("Exploratory NDVI bins (not land-cover classes):")
    for label, count in zip(labels, counts):
        pct = 100 * count / total if total else 0
        print(f"{label:14s}: {count:>12,} pixels ({pct:5.2f}%)")
else:
    print("Skipped: set the correct path and run the raster-loading cell.")


## 8. Conclusions and quality checks

### Once the notebook runs, you can report
- the exported raster's dimensions, CRS, NoData value, and observed pixel-value distribution;
- how NDVI varies spatially and statistically in this scene;
- where relative NDVI is higher or lower.

### You cannot conclude from one NDVI image alone
- exact crop types or species;
- crop yield or soil fertility;
- that every low-NDVI pixel is definitely a building, bare soil, water, or unhealthy vegetation;
- vegetation change over time (that requires comparable dates and careful masking);
- accurate vegetation area without a study boundary, valid-data mask, and defined method.

### Before reporting
1. Confirm the exported GeoTIFF has the expected CRS and 10 m pixel size.
2. Inspect cloud and shadow contamination.
3. Confirm B08 and B04 are from the same Sentinel-2 acquisition.
4. Check reflectance scaling and zero-denominator handling.
5. Keep the QGIS project and exported raster in a stable project folder.

## Next: Notebook 02
**Cloud masking and vegetation mapping:** load the Sentinel-2 Scene Classification Layer (SCL), mask cloud/shadow pixels, clip to a defined Anantapur study boundary, and compare NDVI statistics before and after masking. Then we can use a second date for cautious change analysis.
